# 7Q5A - lanreotide peptide nanotube

Lanreotide is a synthetic cyclic octapeptide (nine residues in the entry, two of
them non-standard) that self-assembles into a monodisperse nanotube. The deposited
biological assembly is an 800-chain slab of the tube wall.

This is the hard case for a structure-derived model: the subunit is tiny, so the
default interface-detection cutoffs see only the tight intra-dimer contact and the
whole design falls apart into 400 disconnected dimers.

Run without ProAffinity: `predict_affinity` is left at its default `False`.

In [ ]:
# Path handling (standard library)
from pathlib import Path

# Core imports
import ionerdss as ion
from ionerdss import build_system_from_pdb

# For visualizations
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
pdb_id = "7q5a"

# Build the system using simplified API
# This takes ~20 s for the 800-chain biological assembly.
system = build_system_from_pdb(
    source=pdb_id,
    workspace_path=f"{pdb_id}_dir",

    # The asymmetric unit is 8 chains = 4 dimers with no dimer-dimer contact,
    # so the tube contacts only exist in the biological assembly.
    pdb_file_format="bioassembly1",

    # THE key parameters for this system. Each chain is a 9-residue peptide, so
    # a real dimer-dimer contact involves only a couple of residues and the
    # ioNERDSS defaults (0.6 nm / 3 residues) reject all of them: the design
    # comes back as 400 disconnected dimers, 2 reactions, max complex size 2.
    #   0.4 nm -> no interfaces at all
    #   0.6 nm -> 2 interfaces, 2 reactions  (dimers only)
    #   0.8 nm -> 6 interfaces, 4 reactions  (dimer + inter-dimer contacts)
    #   1.0 nm -> 14 interfaces, 9 reactions (over-detected)
    # 0.8 nm is the usable choice, but note it is still not the whole tube:
    # the designed contact graph breaks into ~27 components of ~30 peptides
    # each, i.e. ioNERDSS recovers the contacts within a ring of the tube wall
    # but not the weaker contacts that stack ring on ring.
    interface_detect_distance_cutoff=0.8,
    interface_detect_n_residue_cutoff=2,

    chain_grouping_seq_threshold=0.5,

    # A tube is not a sphere: leave the ring regularizer off.
    is_on_sphere=False,

    # 500 peptides in a 150 nm box, ~250 uM -- lanreotide assembles in the
    # high-uM to mM range.
    nerdss_water_box=[150.0, 150.0, 150.0],
    nerdss_total_molecule_count=500,
    nerdss_n_itr=500000,

    # Must be >= nerdss_total_molecule_count. With transition_matrix_size=60
    # this exact model segfaults NERDSS the moment a complex passes 60 subunits.
    count_transition=True,
    transition_matrix_size=500,
)

In [ ]:
# List all generated files
workspace_path = Path(f"{pdb_id}_dir")

print("Generated Files:")

print("\n NERDSS Input Files:")
nerdss_dir = workspace_path / "nerdss_files"
if nerdss_dir.exists():
    for file in sorted(nerdss_dir.glob("*.mol")) + sorted(nerdss_dir.glob("*.inp")):
        size = file.stat().st_size / 1024  # KB
        print(f"  nerdss_files/{file.name:<30} ({size:>6.1f} KB)")

print("\n System Data:")
outputs_dir = workspace_path / "outputs" / "systems"
if outputs_dir.exists():
    for file in sorted(outputs_dir.glob("*.json")):
        size = file.stat().st_size / 1024  # KB
        print(f"  outputs/systems/{file.name:<27} ({size:>6.1f} KB)")

print("\n System Builder Log:")
logs_dir = workspace_path / "logs"
if logs_dir.exists():
    for file in sorted(logs_dir.glob("*.log")):
        size = file.stat().st_size / 1024  # KB
        print(f"  logs/{file.name:<38} ({size:>6.1f} KB)")

# The binding reactions ioNERDSS derived from the structure
print("\n Reaction network:")
print((nerdss_dir / "parms.inp").read_text().split("start reactions")[1])

In [ ]:
# run NERDSS with subprocess
import subprocess

# Check if NERDSS is available
# nerdss_cmd should be replaced with the actual path to the NERDSS executable
nerdss_cmd = "PATH_TO_NERDSS_REPO/bin/nerdss"
nerdss_path = Path(nerdss_cmd).expanduser() # replaces tilde with appropriate user home path

if nerdss_path.exists():

    # Run NERDSS
    result = subprocess.run(
        f"{nerdss_cmd} -f parms.inp",
        shell=True,
        cwd=f"{pdb_id}_dir/nerdss_files",
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print("NERDSS simulation completed!")
        print(f"\nCheck {pdb_id}_dir/nerdss_files/ for output files")
    else:
        # returncode 139 here almost always means a complex grew past
        # transition_matrix_size -- NERDSS indexes its transition matrix by
        # complex size without a bounds check, so keep that value >= the
        # total molecule count.
        print(f"NERDSS simulation failed (returncode {result.returncode})")
        print(result.stderr[:500])
else:
    print("NERDSS not found at:", nerdss_cmd)

In [ ]:
# Initialize Analyzer with NERDSS output directory
analysis = ion.Analyzer(f"{pdb_id}_dir")

print(f"Found {len(analysis.simulations)} simulation(s)")
for i, sim in enumerate(analysis.simulations):
    print(f"  [{i}] Simulation ID: {sim.id}")

sim = analysis.get_simulation(0)

complex_compositions = [{"A": n} for n in [1, 2, 3, 5, 10, 20, 50, 100]]
time, counts = sim.get_time_series(complex_compositions)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, comp in enumerate(complex_compositions):
    axes[0].plot(time, counts[i], label=f"A{comp['A']}")
axes[0].set_xlabel("time (s)")
axes[0].set_ylabel("copy number")
axes[0].set_title("7Q5A: complex counts")
axes[0].legend(ncol=2, fontsize=8)

# A nanotube has no closure size, so the growth curve is the whole story:
# it should keep climbing rather than plateau at a preferred stoichiometry.
t_max, largest = sim.get_largest_size_time_series()
t_avg, average = sim.get_average_size_time_series()
axes[1].plot(t_max, largest, label="largest complex")
axes[1].plot(t_avg, average, label="average complex")
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("subunits per complex")
axes[1].set_title("7Q5A: assembly growth")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"largest complex reached: {np.nanmax(largest):.0f} of 500 peptides")
print(f"free peptides remaining: {counts[0][-1]:.0f} of 500")